In [1]:
 !pip install nltk
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

Defaulting to user installation because normal site-packages is not writeable


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
import re
import ast
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [3]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [4]:
movies.head(5)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [5]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [6]:
credits.columns

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')

In [7]:
movies.shape,credits.shape

((4803, 20), (4803, 4))

In [8]:
movies=movies.merge(credits,on='title')

In [9]:
df=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [10]:
df.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [11]:
df.dropna(inplace=True)

In [12]:
def fetch_genres(text):
    l=[]
    for i in ast.literal_eval(text):
        l.append(i['name'])

    return l

In [13]:
df['genres'].apply(fetch_genres)

0       [Action, Adventure, Fantasy, Science Fiction]
1                        [Adventure, Fantasy, Action]
2                          [Action, Adventure, Crime]
3                    [Action, Crime, Drama, Thriller]
4                [Action, Adventure, Science Fiction]
                            ...                      
4804                        [Action, Crime, Thriller]
4805                                [Comedy, Romance]
4806               [Comedy, Drama, Romance, TV Movie]
4807                                               []
4808                                    [Documentary]
Name: genres, Length: 4806, dtype: object

In [14]:
df['genres']=df['genres'].apply(fetch_genres)

In [15]:
def fetch_keywords(text):
    l=[]
    for i in ast.literal_eval(text):
        l.append(i['name'])

    return l

In [16]:
df['keywords'].apply(fetch_keywords)

0       [culture clash, future, space war, space colon...
1       [ocean, drug abuse, exotic island, east india ...
2       [spy, based on novel, secret agent, sequel, mi...
3       [dc comics, crime fighter, terrorist, secret i...
4       [based on novel, mars, medallion, space travel...
                              ...                        
4804    [united states–mexico barrier, legs, arms, pap...
4805                                                   []
4806    [date, love at first sight, narration, investi...
4807                                                   []
4808            [obsession, camcorder, crush, dream girl]
Name: keywords, Length: 4806, dtype: object

In [17]:
df['keywords']=df['keywords'].apply(fetch_keywords)

In [18]:
def fetch_cast(text):
    l=[]
    counter=0
    for i in ast.literal_eval(text):
        if counter!=3:
            l.append(i['name'])
            counter+=1
        else:
            break

    return l

In [19]:
df['cast'].apply(fetch_cast)

0        [Sam Worthington, Zoe Saldana, Sigourney Weaver]
1           [Johnny Depp, Orlando Bloom, Keira Knightley]
2            [Daniel Craig, Christoph Waltz, Léa Seydoux]
3            [Christian Bale, Michael Caine, Gary Oldman]
4          [Taylor Kitsch, Lynn Collins, Samantha Morton]
                              ...                        
4804    [Carlos Gallardo, Jaime de Hoyos, Peter Marqua...
4805         [Edward Burns, Kerry Bishé, Marsha Dietlein]
4806           [Eric Mabius, Kristin Booth, Crystal Lowe]
4807            [Daniel Henney, Eliza Coupe, Bill Paxton]
4808    [Drew Barrymore, Brian Herzlinger, Corey Feldman]
Name: cast, Length: 4806, dtype: object

In [20]:
df['cast']=df['cast'].apply(fetch_cast)

In [21]:
def fetch_director(text):
    l=[]
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            l.append(i['name'])
        
    return l

In [22]:
df['crew'].apply(fetch_director)

0                                [James Cameron]
1                               [Gore Verbinski]
2                                   [Sam Mendes]
3                            [Christopher Nolan]
4                               [Andrew Stanton]
                          ...                   
4804                          [Robert Rodriguez]
4805                              [Edward Burns]
4806                               [Scott Smith]
4807                               [Daniel Hsia]
4808    [Brian Herzlinger, Jon Gunn, Brett Winn]
Name: crew, Length: 4806, dtype: object

In [23]:
df['crew']=df['crew'].apply(fetch_director)

In [24]:
df['overview'].apply(lambda x: x.split())

0       [In, the, 22nd, century,, a, paraplegic, Marin...
1       [Captain, Barbossa,, long, believed, to, be, d...
2       [A, cryptic, message, from, Bond’s, past, send...
3       [Following, the, death, of, District, Attorney...
4       [John, Carter, is, a, war-weary,, former, mili...
                              ...                        
4804    [El, Mariachi, just, wants, to, play, his, gui...
4805    [A, newlywed, couple's, honeymoon, is, upended...
4806    ["Signed,, Sealed,, Delivered", introduces, a,...
4807    [When, ambitious, New, York, attorney, Sam, is...
4808    [Ever, since, the, second, grade, when, he, fi...
Name: overview, Length: 4806, dtype: object

In [25]:
df['overview']=df['overview'].apply(lambda x: x.split())

In [26]:
df['tags']=df['overview']+df['genres']+df['keywords']+df['cast']+df['crew']

In [27]:
df.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [28]:
data = df[['movie_id', 'title', 'tags']].copy()
data = data.merge(
    movies[['movie_id', 'vote_average', 'vote_count', 'popularity']],
    on='movie_id',
    how='left'
)
data.head()

,movie_id,title,tags,vote_average,vote_count,popularity
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...",7.2,11800,150.437577
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...",6.9,4500,139.082615
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...",6.3,4466,107.376788
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...",7.6,9106,112.312950
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...",6.1,2124,43.926995


In [29]:
def clean_tags(x):
    if isinstance(x, (list, tuple)):
        return " ".join(
            str(i).replace(" ", "")
            for i in x
            if i is not None
        )
    if isinstance(x, str):
        return x
    return ""
data['tags'] = data['tags'].apply(clean_tags)


In [30]:
data.head()

,movie_id,title,tags,vote_average,vote_count,popularity
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",7.2,11800,150.437577
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",6.9,4500,139.082615
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,6.3,4466,107.376788
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,7.6,9106,112.312950
4,49529,John Carter,"John Carter is a war-weary, former military ca...",6.1,2124,43.926995


In [31]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def text_preprocess(text):
    clean = []
    for word in text.split():
        word = word.lower()
        if word not in stop_words:
            word = lemmatizer.lemmatize(word)
            clean.append(word)
    return " ".join(clean)

In [32]:
data['tags']=data['tags'].apply(text_preprocess)

In [33]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2)
)
tfidf

TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')

In [34]:
vectors = tfidf.fit_transform(data['tags'])
vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 169668 stored elements and shape (4818, 10000)>

In [35]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

data[['rating_score', 'popularity_score']] = scaler.fit_transform(
    data[['vote_average', 'popularity']]
)

In [36]:
data[['title', 'vote_average', 'popularity',
      'rating_score', 'popularity_score']].head()

,title,vote_average,popularity,rating_score,popularity_score
0,Avatar,7.2,150.437577,0.72,0.171815
1,Pirates of the Caribbean: At World's End,6.9,139.082615,0.69,0.158846
2,Spectre,6.3,107.376788,0.63,0.122635
3,The Dark Knight Rises,7.6,112.312950,0.76,0.128272
4,John Carter,6.1,43.926995,0.61,0.050169


In [37]:
def recommend(movie):
    movie_index = data[data['title'] == movie].index[0]
    distance = cosine_similarity(
        vectors[movie_index],
        vectors
    ).flatten()
    movie_list = sorted(
        list(enumerate(distance)),
        reverse=True,
        key=lambda x: x[1]
    )[1:21]
    recommendations = []
    for i in movie_list:
        movie_idx = i[0]
        content_score = float(i[1])
        rating_score = float(data.iloc[movie_idx]['rating_score'])
        popularity_score = float(data.iloc[movie_idx]['popularity_score'])
        final_score = (
            0.75 * content_score +
            0.20 * rating_score +
            0.05 * popularity_score
        )
        recommendations.append({
            'movie_id': data.iloc[movie_idx]['movie_id'],
            'title': data.iloc[movie_idx]['title'],
            'similarity': round(content_score, 3),
            'rating': round(float(data.iloc[movie_idx]['vote_average']), 1),
            'popularity': round(float(data.iloc[movie_idx]['popularity']), 1),
            'score': round(final_score, 3)
        })
    recommendations = sorted(
        recommendations,
        key=lambda x: x['score'],
        reverse=True
    )
    return recommendations[:10]

In [38]:
test_movies = [
    'Iron Man',
    'Harry Potter and the Chamber of Secrets',
]
for movie in test_movies:
    print(f"\nRecommendations for: {movie}")
    for rec in recommend(movie):
        print(rec)


Recommendations for: Iron Man
{'movie_id': np.int64(10138), 'title': 'Iron Man 2', 'similarity': 0.463, 'rating': 6.6, 'popularity': 77.3, 'score': 0.484}
{'movie_id': np.int64(68721), 'title': 'Iron Man 3', 'similarity': 0.411, 'rating': 6.8, 'popularity': 77.7, 'score': 0.449}
{'movie_id': np.int64(99861), 'title': 'Avengers: Age of Ultron', 'similarity': 0.256, 'rating': 7.3, 'popularity': 134.3, 'score': 0.346}
{'movie_id': np.int64(24428), 'title': 'The Avengers', 'similarity': 0.212, 'rating': 7.4, 'popularity': 144.4, 'score': 0.315}
{'movie_id': np.int64(118340), 'title': 'Guardians of the Galaxy', 'similarity': 0.168, 'rating': 7.9, 'popularity': 481.1, 'score': 0.311}
{'movie_id': np.int64(102899), 'title': 'Ant-Man', 'similarity': 0.214, 'rating': 7.0, 'popularity': 120.1, 'score': 0.307}
{'movie_id': np.int64(36657), 'title': 'X-Men', 'similarity': 0.208, 'rating': 6.8, 'popularity': 4.7, 'score': 0.292}
{'movie_id': np.int64(271110), 'title': 'Captain America: Civil War',

In [39]:
pickle.dump(
    data.to_dict(),
    open('movie_dict.pkl', mode='wb')
)
pickle.dump(
    vectors,
    open('tfidf_vectors.pkl', mode='wb')
)
pickle.dump(
    tfidf,
    open('tfidf.pkl', mode='wb')
)